# 技能2 · Day 4 上机：企业级架构参考设计 + 行动研究

**版本**：v5.0 学习材料包（技能2收官）
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 设计CDP核心schema（Identity/Event/Segment/Profile四层），基于Segment Spec真实公开规范
2. 用 **networkx + TOGAF/ArchiMate** 建模企业AI架构四层依赖图，分析关键依赖路径
3. 用 **pandas** 分析行动研究（Action Research）迭代循环KPI，理解"研究即干预"
4. 把Day1-3整合为**营销中心AI原生参考架构**，定位为DSR artifact
5. 理解**天道推演×企业架构**的同构关系：架构设计即沙盘推演

## 真实库
- **pydantic**：CDP schema建模（对标Twilio Segment数据模型）
- **networkx + matplotlib**：架构依赖图建模与可视化（对标TOGAF/ArchiMate）
- **pandas**：行动研究迭代KPI分析
- **真实规范**：Segment Spec (https://segment.com/docs/spec/)


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 所有库（pydantic/networkx/matplotlib/pandas）均为本地可用库，不需要API Key。

In [ ]:
# !pip install pydantic networkx matplotlib pandas -q

import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timezone
from typing import Any, Optional
from pydantic import BaseModel, Field

import networkx as nx
import matplotlib
matplotlib.use('Agg')  # 非交互式后端，适合脚本运行
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

print("环境就绪")
print(f"  pydantic: CDP schema建模")
print(f"  networkx: 架构依赖图")
print(f"  matplotlib: 架构图可视化")
print(f"  pandas: 行动研究KPI分析")


## 1. CDP身份层设计（对标Segment Identify Spec）

**CDP（客户数据平台）** 是AI原生架构的数据基础设施。本节用pydantic设计CDP四层schema的第一层--身份层(Identity)。

**Segment Identify Spec** 定义了三个核心字段：
- `userId`：已知用户的唯一标识（登录后）
- `anonymousId`：匿名用户的唯一标识（首次访问）
- `traits`：用户属性字典（年龄/性别/兴趣等自由格式）

**营销映射**：Identity层是CDP的基础--所有用户行为数据都挂在Identity上，Agent通过Identity查询用户画像。

参考：https://segment.com/docs/connections/spec/identify/


In [ ]:
# TODO 1：CDP身份层设计 -- 用pydantic设计Identity模型
# 提示：对标Segment Identify Spec
#   核心字段：user_id (str), anonymous_id (str), traits (dict[str, Any])
#   可选字段：email (str|None), created_at (datetime)
#   用Field添加描述和默认值
# 要求：定义Identity(BaseModel)，实例化一个营销用户，打印模型

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：实例化并打印
# identity = Identity(...)
# print(identity)


## 2. CDP事件层设计（对标Segment Track Spec）

**Segment Track Spec** 定义了用户行为事件的三个核心字段：
- `userId`：触发事件的用户标识
- `event`：事件名称（如"Order Completed"、"Product Viewed"）
- `properties`：事件属性字典（订单金额/产品ID等）

**营销映射**：Event层记录用户与营销触点的所有交互--页面浏览、内容互动、购买行为。这些事件流是实时画像更新和Agent决策的数据源。

参考：https://segment.com/docs/connections/spec/track/


In [ ]:
# TODO 2：CDP事件层设计 -- 用pydantic设计Event模型
# 提示：对标Segment Track Spec
#   核心字段：user_id (str), event_name (str), properties (dict[str, Any])
#   可选字段：event_id (str|None), timestamp (datetime), context (dict|None)
#   event_name用Literal约束为营销事件类型（如"ProductViewed", "OrderCompleted"等）
# 要求：定义Event(BaseModel)，实例化一个"用户浏览产品"事件，打印模型

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：实例化并打印
# event = Event(...)
# print(event)


## 3. CDP分群层 + 画像层设计

**分群层(Segment)**：将用户按条件分组（如"高价值客户"、"流失风险用户"），支持营销Agent按分群执行差异化策略。

**画像层(Profile)**：整合Identity的traits、Event的聚合统计、Segment的归属信息，形成用户的完整360度画像。在AI原生架构中，Profile还包含向量化表示（embedding），支持语义级别的用户理解。

这两层共同构成CDP的"AI激活层"--Agent通过Profile API获取用户数据，通过Segment API获取目标人群。


In [ ]:
# TODO 3：CDP分群层 + 画像层设计 -- 用pydantic设计Segment和Profile模型
# 提示：
#   Segment: segment_id (str), name (str), criteria (str描述分群条件),
#            user_ids (list[str]), created_at (datetime)
#   Profile: user_id (str), traits (dict), event_count (int),
#            segments (list[str]), embedding (list[float]模拟向量),
#            last_updated (datetime)
# 要求：定义Segment和Profile，实例化"高价值客户"分群和一个用户画像，打印

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：实例化并打印
# segment = Segment(...)
# profile = Profile(...)
# print(segment)
# print(profile)


## 4. 企业架构依赖图（TOGAF四层 + networkx）

用 **networkx DiGraph** 建模企业AI架构的组件依赖关系，对标 **TOGAF/ArchiMate** 的四层架构域：

| TOGAF架构域 | AI原生架构组件 |
|-------------|---------------|
| 业务架构 | MarketingCampaign, CustomerJourney |
| 应用架构 | InsightAgent, ContentAgent, PlacementAgent, AnalyticsAgent, CoordinatorAgent |
| 数据架构 | CDP(Identity/Event/Segment/Profile), VectorDB, KnowledgeGraph |
| 技术架构 | LLMService, RAGEngine, DataPipeline, InferenceServer |

**任务**：构建架构依赖图，计算节点数/边数，分析关键依赖路径（如"数据层故障如何影响业务层"）。

参考：https://www.opengroup.org/capabilities/togaf


In [ ]:
# TODO 4：企业架构依赖图 -- 用networkx DiGraph建模TOGAF四层架构
# 提示：
#   1. 创建DiGraph，按TOGAF四层添加节点（业务/应用/数据/技术）
#   2. 每个节点设置layer属性（"business"/"application"/"data"/"technology"）
#   3. 添加依赖边（如CDP->InsightAgent表示Agent依赖CDP数据）
#   4. 计算并打印：节点数、边数、从"DataPipeline"到"MarketingCampaign"的最短路径
# 要求：构建>=12个节点、>=15条边的架构依赖图，分析关键依赖路径

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：打印图的基本信息和关键路径
# print(f"节点数: {G.number_of_nodes()}")
# print(f"边数: {G.number_of_edges()}")
# path = nx.shortest_path(G, "DataPipeline", "MarketingCampaign")
# print(f"关键路径: {path}")


## 5. CDP数据流图可视化

用 **networkx + matplotlib** 可视化CDP在营销中心的数据流：

```
数据源(Website/App/CRM)
  ↓ Identify/Track API
CDP身份层+事件层
  ↓ 实时流处理
CDP画像层+分群层
  ↓ Profile/Segment API
Agent编排层(洞察/内容/投放/分析)
  ↓ 营销动作
触达渠道(邮件/短信/广告)
```

**任务**：构建CDP数据流图，用matplotlib绘制可视化，标注数据流向和组件类型。


In [ ]:
# TODO 5：CDP数据流图可视化 -- 用networkx+matplotlib绘制
# 提示：
#   1. 构建CDP数据流图（数据源->CDP四层->Agent层->触达渠道）
#   2. 用node_type属性区分组件类型（"source"/"cdp"/"agent"/"channel"）
#   3. 用matplotlib绘制图，不同类型用不同颜色
#   4. 用spring_layout或自定义布局
# 要求：绘制>=10个节点的CDP数据流图，保存或显示图片，打印节点类型分布

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：打印节点类型分布
# for nt in ["source", "cdp", "agent", "channel"]:
#     count = sum(1 for n in flow_g.nodes if flow_g.nodes[n].get("node_type") == nt)
#     print(f"  {nt}: {count}个节点")


## 6. 行动研究迭代分析（Plan/Act/Observe/Reflect）

用 **pandas** 分析行动研究的迭代循环数据。行动研究（Action Research）采用Susman & Evered (1978)的五步螺旋：诊断->规划->行动->评估->反思。

**数据说明**：以下KPI数据基于真实行动研究文献报告的改善幅度区间构建（非单一案例精确数据）：
- 决策时间：AI部署后降低30%-60%
- 决策质量：提升1.0-2.5分（10分制）
- AI使用率：从10%->70%（随迭代轮次增长）
- 团队满意度：首轮下降0.3-0.5（学习曲线），后续回升0.5-1.0

来源：Susman & Evered (1978); Kemmis et al. (2014); Coughlan & Coghlan (2002)

**任务**：加载行动研究迭代数据，分析各轮KPI变化趋势，计算改善幅度。


In [ ]:
# TODO 6：行动研究迭代分析 -- 用pandas分析Plan/Act/Observe/Reflect多轮KPI
# 提示：
#   1. 创建DataFrame，包含4轮行动研究迭代，每轮有4个KPI：
#      decision_time_min(决策时间), decision_quality(决策质量1-10),
#      ai_usage_rate(AI使用率%), team_satisfaction(团队满意度1-5)
#   2. KPI改善幅度参考真实文献：决策时间降30-60%，质量升1-2.5分，
#      AI使用率10%->70%，满意度先降后升
#   3. 计算每轮相对基线(Round 0)的改善幅度
#   4. 分析趋势：哪轮改善最大？哪个KPI改善最显著？
# 要求：构建4轮迭代DataFrame，计算改善幅度，打印趋势分析

# ===== 你的代码 =====

# raise NotImplementedError

# 验证：打印迭代数据和改善幅度
# print(ar_df)
# print(improvement_df)


## 7. 反思与前沿

### 反思问题
1. 你的CDP schema设计是否覆盖了营销Agent需要的所有数据？缺什么？
2. 架构依赖图中哪条路径最脆弱？如果CDP故障，哪些Agent会受影响？
3. 行动研究的KPI改善幅度是否符合预期？哪轮改善最大？为什么？
4. 用天道推演视角分析：你在架构设计时推演了几个方案？每个方案的3层未来走向是什么？

### 2026前沿：天道推演×企业架构 + DSR + 可复现研究

**天道推演×企业架构**：企业架构设计本质上是对组织的沙盘推演--架构师在意识中构建多个架构方案的平行世界，模拟其未来走向，选择最优路径。天道推演的五项能力（局势感知/因果链追踪/沙盘模拟/概率评估/最优路径推荐）与企业架构设计的五个阶段（现状审计/依赖分析/方案模拟/风险评估/选型推荐）同构。

**多Agent仿真×架构验证**：2026年前沿趋势是用多Agent仿真验证架构设计--在部署真实系统前，先用多Agent仿真模拟架构运行情况，预测消息传递延迟、资源竞争、故障传播路径。

**DSR + 可复现研究**：企业架构设计作为DSR artifact，行动研究作为评估方法。你的架构依赖图（networkx）+ CDP schema（pydantic）+ 行动研究KPI（pandas）全部用代码定义，他人可独立复现。

### 天道推演×企业架构 同构映射表

| 天道推演能力 | 企业架构设计对应 | 本Day上机对应 |
|-------------|----------------|-------------|
| 局势感知 | 现有架构审计 | CDP schema现状分析 |
| 因果链追踪 | 架构依赖图分析 | networkx DAG + shortest_path |
| 沙盘模拟(3层) | 多架构方案并行模拟 | CDP数据流图可视化 |
| 概率评估 | 方案风险/成本/收益概率 | 行动研究KPI不确定性 |
| 最优路径推荐 | 架构选型建议 | 营销中心参考架构 |

> 深入阅读见 `reading.md` 的TOGAF、行动研究、DSR条目。
